# 02 — Overfit One Batch (Sanity Check)

**Sprint 3 — Checkpoint 3.2**

Test critico: entrenar con solo 4 imagenes durante 100 epochs.

**Criterios de exito:**
- Training Loss baja a ~0.0
- Training Accuracy sube a 100%
- Si no pasa, hay un bug en el codigo (no en los datos)

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

## 1. Preparar modelo, batch y criterion

In [ ]:
import torch
import torch.nn as nn

from src.config import cfg
from src.model import Simple3DCNN
from src.dataset import get_dataloader
from src.train import compute_class_weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.manual_seed(cfg.RANDOM_SEED)

model = Simple3DCNN().to(device)
class_weights = compute_class_weights().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE)

loader = get_dataloader("train", batch_size=4, num_workers=0, shuffle=False)
single_batch = next(iter(loader))
images = single_batch["image"].to(device)
labels = single_batch["label"].to(device)

print(f"Images shape: {images.shape}")
print(f"Labels: {labels.tolist()}")

## 2. Entrenar 100 epochs sobre el batch

In [ ]:
num_epochs = 100
history = {"loss": [], "accuracy": []}

model.train()
for epoch in range(1, num_epochs + 1):
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    preds = outputs.argmax(dim=1)
    acc = (preds == labels).float().mean().item()

    history["loss"].append(loss.item())
    history["accuracy"].append(acc)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{num_epochs} — Loss: {loss.item():.4f}  Acc: {acc:.2%}")

print(f"\nFinal — Loss: {history['loss'][-1]:.6f}  Acc: {history['accuracy'][-1]:.2%}")

## 3. Visualizar curvas de convergencia

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history["loss"], linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Overfit One Batch — Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(history["accuracy"], linewidth=2, color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Overfit One Batch — Accuracy")
ax2.set_ylim(-0.05, 1.05)
ax2.axhline(y=1.0, color="red", linestyle="--", alpha=0.5, label="100%")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Assert final

In [ ]:
final_loss = history["loss"][-1]
final_acc = history["accuracy"][-1]

print(f"Final Loss:     {final_loss:.6f}  (debe ser < 0.05)")
print(f"Final Accuracy: {final_acc:.2%}    (debe ser 100%)")

assert final_loss < 0.05, f"Loss demasiado alta: {final_loss:.4f}"
assert final_acc == 1.0, f"Accuracy no es 100%: {final_acc:.2%}"

print("\n=== CHECKPOINT 3.2 PASSED ===")